# LangChain 2026 — Revision Notebook 

**Landscape -> Environment Setup -> Models & Messages -> Prompt Templates**

This single notebook is a complete revision of everything covered so far. Every concept has an
explanation, an analogy, a diagram, and real runnable code with printed output -- run every cell
top to bottom to refresh the whole foundation in one sitting.

```mermaid
graph LR
    A[Part 1: Landscape] --> B[Part 2: Environment Setup]
    B --> C[Part 3: Models and Messages]
    C --> D[Part 4: Prompt Templates]
    D --> E[Next: Structured Output, Tools, Agents...]
```


---
# PART 1 — The LangChain Family and the Agent Model

## 1.1 The Four Products: LangChain, LangGraph, LangSmith, Deep Agents

Four products, same team, four different jobs. None of them compete with each other -- you use
as many as your project needs.

```mermaid
graph LR
    A[LangGraph<br/>Foundation] --> B[LangChain create_agent<br/>THIS COURSE]
    B --> C[Deep Agents<br/>Batteries included]
    D[LangSmith] -.watches all three.-> A
    D -.-> B
    D -.-> C
```

**Analogy — a construction site:** LangGraph is the foundation and structural steel, you rarely
touch it directly. LangChain's `create_agent` is the prefabricated rooms going up on that
foundation -- pre-built, still customizable. Deep Agents is the fully furnished show-home next
door, move-in ready. LangSmith is completely different from the other three: it's the site
inspector standing at a desk with cameras pointed at everything else -- because an agent decides
what to do at *runtime*, so you can't just read the code to know what it actually did. Only a
recorded trace tells you that.

| Product | What it's for |
|---|---|
| **LangGraph** | Low-level orchestration -- the engine underneath everything else |
| **LangChain** | `create_agent` -- a highly configurable harness. **This entire course.** |
| **Deep Agents** | Batteries-included harness -- planning, filesystem, subagents pre-wired |
| **LangSmith** | Observability -- traces, debugs, evaluates agents built with ANY of the other three |


## 1.2 The Single Most Important Sentence in This Course

> **"An agent is a model calling tools in a loop until a given task is complete.**
> **A harness is everything around that loop."**

```mermaid
graph LR
    A["The Model<br/>(raw power, stuck)"] -->|"+ create_agent<br/>(the harness)"| B["The Agent<br/>(model + prompt + tools + middleware)"]
```

**Analogy — a Formula 1 car:** a bare engine on a workbench is extraordinary engineering that
can't take you anywhere. It needs a chassis, tires, and a pit crew before it becomes a car that
wins races. The model is the engine. `create_agent` is everything that turns that engine into a
car -- what it can see (system prompt), what it can reach for (tools), what checkpoints govern
its decisions (middleware).

**Why this matters for the rest of the course:** almost everything from here forward -- RAG,
memory, multi-agent systems -- is just a *different configuration of the harness* around the
same underlying model. The model never changes. Only what you give it access to does.


In [1]:
# A quick way to hold onto Part 1's core idea -- print it as a reminder before we start coding.
core_idea = "An agent is a model calling tools in a loop until a given task is complete."
harness_components = ["system prompt", "tools", "middleware"]

print(core_idea)
print()
print("The harness =", " + ".join(harness_components))


An agent is a model calling tools in a loop until a given task is complete.

The harness = system prompt + tools + middleware


## 1.3 A Real Timeline

```mermaid
graph LR
    A["Oct 2022<br/>LangChain launches"] --> B["Dec 2022<br/>First ReAct agents"]
    B --> C["Feb 2024<br/>LangGraph released"]
    C --> D["Oct 2024<br/>LangGraph preferred"]
    D --> E["Oct 20 2025<br/>v1.0 - create_agent"]
    E --> F["Mar 15 2026<br/>Deep Agents"]
```

| Date | What happened |
|---|---|
| Oct 2022 | LangChain launches — LLM abstractions + "Chains" |
| Dec 2022 | First ReAct-based agents (model generates JSON -> hand-parsed into tool calls) |
| Feb 2024 | LangGraph released — the low-level orchestration layer that was missing |
| Oct 2024 | LangGraph becomes preferred for anything beyond a single call; old chains deprecated |
| **Oct 20, 2025** | **LangChain v1.0** — one agent abstraction (`create_agent`) replaces everything |
| **Mar 15, 2026** | **Deep Agents released** |

**The practical rule:** if a 2026-dated tutorial still shows `AgentExecutor` or
`initialize_agent`, it predates October 2025 -- that functionality now lives in a separate
`langchain-classic` package, deliberately kept apart from the main framework.


## 1.4 The Three-Tier Picture

```mermaid
graph LR
    A["LangGraph<br/>Whole spices,<br/>grind your own"] --> B["LangChain create_agent<br/>Pre-mixed masala,<br/>still adjustable"]
    B --> C["Deep Agents<br/>Dish already cooked,<br/>still season to taste"]
```

LangGraph (most control, most effort) -> LangChain `create_agent` (**this course**) -> Deep
Agents (batteries-included). All three are built on LangGraph. You do **not** need to know
LangGraph to use `create_agent` -- that's a deliberate design choice, not a shortcut.

### Quick self-check (answer before moving on)
1. If you need to *build* an agent quickly with a lot of customization, which product do you reach for?
2. If your agent misbehaves in production and you need to know exactly what happened on one run, which product?
3. Fill in the blank: "An agent is a model calling ___ in a ___ until a task is complete."


In [2]:
# Self-check answers -- run this AFTER you've thought about the three questions above
answers = {
    1: "LangChain's create_agent (or Deep Agents if you want batteries-included)",
    2: "LangSmith",
    3: "tools ... loop",
}
for q, a in answers.items():
    print(f"Q{q}: {a}")


Q1: LangChain's create_agent (or Deep Agents if you want batteries-included)
Q2: LangSmith
Q3: tools ... loop


---
# PART 2 — Environment Setup

## 2.1 Installing and Configuring

Every notebook in this course assumes this environment already exists.

```mermaid
graph LR
    A[.env file] -->|load_dotenv| B[Your Code]
    B -->|never printed or committed| C[GitHub Repo]
    A -.gitignore excludes.-> C
```

**Analogy — the backstage pass:** your `.env` file is the backstage pass drawer at a venue. Your
code knows a valid pass exists and can use it, but the pass itself never walks out on stage
where the audience -- anyone reading your GitHub repo -- can see it.


In [ ]:
# Run these in a terminal (or a Colab cell prefixed with !), not as plain Python:
#
#   uv init langchain-course && cd langchain-course
#   uv add langchain langchain-openai langchain-community langgraph python-dotenv
#   uv add langchain-mcp-adapters langchain-chroma chromadb pypdf
#
# In Colab specifically, install directly:
# !pip install -q langchain langchain-openai langchain-community langgraph python-dotenv langchain-chroma chromadb

print("Run the install command above once per Colab session (Colab resets packages each time).")


In [ ]:
# Sanity check -- run this at the start of EVERY notebook in this course.
# In Colab: store your key in Colab's Secrets (key icon in the left sidebar), NOT hardcoded here.

import os
from dotenv import load_dotenv

load_dotenv()  # picks up a local .env file if one exists

# Colab-specific: uncomment these two lines if using Colab Secrets instead of a .env file
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Missing OPENAI_API_KEY -- set it via .env or Colab Secrets"
print("Environment OK. Key starts with:", os.environ["OPENAI_API_KEY"][:7] + "...")


**Why the Colab Secrets approach matters:** it keeps the real key out of the notebook file
itself entirely -- the same underlying goal as `.env`, just Colab's own mechanism for it. Never
type `os.environ["OPENAI_API_KEY"] = "sk-..."` directly into a cell with the real key visible --
that's exactly the mistake that leaks a key the moment the notebook is shared.


## 2.2 Proving It All Works: The Official Quickstart

This is the exact snippet from the official docs, unmodified. If it runs, your environment is
genuinely correct.


In [ ]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="openai:gpt-5-mini",   # check docs.langchain.com for the current model catalog
                                   # if this specific model has changed -- the code around it
                                   # never needs to change, only this string.
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result["messages"][-1].content_blocks)


**If this raised an authentication error:** the two most common causes are (1) your API key
isn't actually loaded into the environment yet, or (2) you set it up *after* starting this
notebook session -- restart the runtime so the key is picked up fresh.


---
# PART 3 — Models and Messages

## 3.1 The Standard Model Interface

```mermaid
graph LR
    A["'provider:model' string"] --> B[init_chat_model]
    B --> C[OpenAI - paid]
    B --> D[Anthropic - paid]
    B --> E[Ollama - free, local]
    B --> F[OpenRouter - free tier available]
```

One consistent interface across every provider. Switch providers by changing **one string** --
everything else (`.invoke()`, `.stream()`, `.bind_tools()`) works identically.


In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5-mini")
response = model.invoke("In one line, tell me what LangChain is.")
print(response.content)


### Dashboard dials: parameters on every model call

**Analogy:** dashboard dials in a car. `temperature` is how much creative liberty you allow with
the route. `max_tokens` is a hard fuel ceiling on response length. `timeout` is how long you
wait at a red light before giving up. `max_retries` is how many times you try restarting a car
that won't start.


In [ ]:
deterministic_model = init_chat_model("openai:gpt-5-mini", temperature=0.0, max_tokens=200, timeout=30)
creative_model = init_chat_model("openai:gpt-5-mini", temperature=1.0, max_tokens=200, timeout=30)

prompt = "Describe a sunset in one creative sentence."
print("temperature=0.0:", deterministic_model.invoke(prompt).content)
print("temperature=1.0:", creative_model.invoke(prompt).content)
# Run temperature=0.0 twice -- it comes back nearly identical. temperature=1.0 varies each run.


### Free and paid models, side by side

You are not locked into paid APIs. Ollama runs fully locally (zero cost, needs the model
downloaded first); OpenRouter offers a free-tier router. The interface never changes.


In [ ]:
# PAID: OpenAI, Anthropic, Google -- all through init_chat_model
paid_model = init_chat_model("openai:gpt-5-mini")

# FREE / LOCAL: Ollama -- requires `ollama pull llama3.2` first, runs on your own machine
# free_local_model = init_chat_model("ollama:llama3.2")

# FREE (rate-limited) via OpenRouter's free router -- requires OPENROUTER_API_KEY set
# free_routed_model = init_chat_model("openrouter/free")

print("Same init_chat_model() call, same .invoke() interface, regardless of which one you use.")
print("Only the provider string changes -- that's the entire point of a standard interface.")


## 3.2 Messages: System, Human, AI, Tool

```mermaid
graph TB
    A[Message List] --> B["SystemMessage<br/>director's note"]
    A --> C["HumanMessage<br/>what the user said"]
    A --> D["AIMessage<br/>what the model said"]
    A --> E["ToolMessage<br/>a tool's result"]
```

**Analogy:** a raw prompt string is a note slid under a door -- no way to tell instructions from
questions. A message list stamps every piece with who "said" it.


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a pirate. Answer everything in pirate speak."),
    HumanMessage(content="What's the capital of France?"),
]
response = model.invoke(messages)
print(response.content)
# Notice: the "be a pirate" instruction lives entirely in the SystemMessage --
# never mixed into what the user actually asked.


### A `HumanMessage` can carry optional metadata too

`name` (useful in multi-user conversations to label who's speaking) and `id` (a stable
identifier) are both optional fields worth knowing about.


In [ ]:
human_msg = HumanMessage(
    content="Hi, how are you?",
    name="mayank",   # optional -- who is speaking, useful in group/multi-user chats
    id="user_1234",  # optional -- a stable identifier for this specific message
)
print(human_msg)


## 3.3 The Full `AIMessage` Anatomy

Most tutorials only ever print `.content`. There's far more sitting on the same object.

```mermaid
graph TB
    A[AIMessage] --> B[".text / .content<br/>the reply itself"]
    A --> C[".content_blocks<br/>standardized rich content"]
    A --> D[".tool_calls<br/>any tool requests"]
    A --> E[".id<br/>unique identifier"]
    A --> F[".usage_metadata<br/>real token counts"]
    A --> G[".response_metadata<br/>provider-specific extras"]
```


In [ ]:
response = model.invoke("Explain agentic AI in one sentence.")

print("text:             ", response.text)
print("content_blocks:   ", response.content_blocks)
print("id:               ", response.id)
print("tool_calls:       ", response.tool_calls)
print("usage_metadata:   ", response.usage_metadata)


In [ ]:
# .pprint() / pprint() gives a nicely formatted full view of everything on the object at once
from pprint import pprint
pprint(response)


## 3.4 Streaming: `AIMessageChunk` Objects That Sum With `+`

```mermaid
graph LR
    A[chunk 1] -->|+| B[chunk 2]
    B -->|+| C[chunk 3]
    C --> D["Full AIMessage<br/>identical to .invoke()"]
```

When you stream instead of waiting for the full response, you get a SEQUENCE of
`AIMessageChunk` objects -- and you can literally add them together with `+`.


In [ ]:
chunks = []
full_message = None
for chunk in model.stream("Write one short sentence about the ocean."):
    chunks.append(chunk)
    print(repr(chunk.text), " <- chunk type:", type(chunk).__name__)
    full_message = chunk if full_message is None else full_message + chunk

print()
print("Number of chunks:", len(chunks))
print("Reconstructed full message:", full_message.content)


## 3.5 Batching: Running Multiple Prompts in Parallel

**Analogy — a restaurant kitchen:** `.batch()` sends every order at once and brings them ALL out
together when the last one finishes. `.batch_as_completed()` brings each dish out the moment
it's ready, possibly out of order.


In [ ]:
questions = [
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?",
]

# .batch() -- waits for ALL, returns in the SAME order as input
responses = model.batch(questions)
for q, r in zip(questions, responses):
    print(f"Q: {q}\nA: {r.content[:80]}...\n")


In [ ]:
# .batch_as_completed() -- yields each result AS IT FINISHES, order not guaranteed
print("Results as they complete:")
for idx, response in model.batch_as_completed(questions):
    print(f"[input #{idx}] {questions[idx][:35]}... -> {response.content[:50]}...")


## 3.6 Tool Calls Live Inside the `AIMessage` (a preview of Part 6)

When a model decides it needs to call a tool, that request sits in the exact `.tool_calls`
field from section 3.3 -- nothing runs yet, this is purely a *request*.


In [ ]:
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"Sunny in {location}"

def set_password(new_pass: str) -> str:
    """Set a new password."""
    return "Password changed"

model_with_tools = model.bind_tools([get_weather, set_password])
response = model_with_tools.invoke("Set the password to admin123")

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"ID:   {tool_call['id']}")
# Nothing has executed -- this is the model's REQUEST. Actually running it and completing
# the loop is what create_agent (Part 7) and full Tools coverage (Part 6) build properly.


## 3.7 `ToolMessage`: Completing the Loop by Hand (once, to see the mechanism)

Before `create_agent` automates this (Part 7), it's worth seeing the raw mechanism once:
execute the requested tool yourself, wrap the result in a `ToolMessage`, and send the whole
conversation back to the model.


In [ ]:
from langchain_core.messages import ToolMessage, AIMessage

# Step 1: reuse the actual tool call the model requested back in section 3.6's "response"
requested_call = response.tool_calls[0]
print("Model requested:", requested_call["name"], requested_call["args"])

# Step 2: actually execute it ourselves -- this is the part create_agent will automate later
tool_result = get_weather(**requested_call["args"])
print("Tool executed manually, result:", tool_result)

# Step 3: wrap the result as a ToolMessage, matching the EXACT tool_call_id from the request
tool_message = ToolMessage(content=tool_result, tool_call_id=requested_call["id"])

# Step 4: send the full conversation -- original question, the AI's tool request (as a real
# AIMessage with tool_calls), and the tool's result -- back to the model for a final answer
conversation = [
    HumanMessage("Set the password to admin123"),
    AIMessage(content="", tool_calls=[requested_call]),
    tool_message,
]
final_response = model.invoke(conversation)
print()
print("Final natural-language answer:", final_response.content)
# THIS manual loop -- request, execute, wrap, send back -- is exactly what create_agent
# automates for you starting in Part 7. Seeing it done by hand once makes the automation
# feel like a shortcut for real work, not magic.


## 3.8 `ToolMessage.artifact`: Two Audiences for One Tool Result

`.content` is what the MODEL reads. `.artifact` is extra data your APPLICATION can use, that is
**never sent to the model** -- useful for citation links, document IDs, anything the model
doesn't need but your UI does.


In [ ]:
artifact_example = ToolMessage(
    content="It was the best of times, it was the worst of times.",
    tool_call_id="call_456",
    artifact={"document_id": "doc_123", "page": 0},
)
print("What the MODEL sees:", artifact_example.content)
print("What your APP can use (model never sees this):", artifact_example.artifact)


## 3.9 Multiple Ways to Represent a Conversation

Message objects and plain dictionaries mean the same thing -- dicts are common when history
comes from a database or JSON API.


In [ ]:
# Objects -- what this course has used throughout
conversation_objects = [
    SystemMessage("You are a helpful assistant that translates English to French."),
    HumanMessage("Translate: I love programming."),
]

# Dictionaries -- identical meaning, no imports needed
conversation_dicts = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
]

r1 = model.invoke(conversation_objects)
r2 = model.invoke(conversation_dicts)
print("Object-based:", r1.content)
print("Dict-based:  ", r2.content)


---
# PART 4 — Prompt Templates

## 4.1 Three Ways to Prompt, and When to Use Each

| Shape | Use when |
|---|---|
| Plain text | A single, standalone request; no history; minimal code |
| Message objects | Multi-turn conversations; multimodal content; system instructions |
| Dictionaries | Same as objects, when data arrives from a database or JSON API |


In [ ]:
# Shape 1: plain text -- simplest possible case
response = model.invoke("Write a haiku about spring")
print(response.content)


## 4.2 Building a Reusable Template: `ChatPromptTemplate`

```mermaid
graph LR
    A["ChatPromptTemplate<br/>system: tone, human: topic"] --> B["chain = template | model"]
    C["tone='playful'"] --> B
    D["tone='deadpan'"] -.same template, new values.-> B
```

**Analogy:** a fresh f-string is a napkin you scribble on every time. A `ChatPromptTemplate` is
a resume template with labeled blank fields -- the structure is fixed and trustworthy, only the
specifics change.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

fun_fact_prompt = ChatPromptTemplate.from_messages([
    ("system", "You generate a single surprising fun fact. Tone: {tone}."),
    ("human", "Topic: {topic}"),
])

chain = fun_fact_prompt | model  # the pipe operator: "feed the output of this into that"

for tone, topic in [
    ("playful and silly", "octopuses"),
    ("deadpan and serious", "octopuses"),
]:
    result = chain.invoke({"tone": tone, "topic": topic})
    print(f"[{tone}] -> {result.content}\n")


## 4.3 The Single Most Common Real Bug: Unescaped Curly Braces

If a template's fixed text contains literal curly braces -- often from pasting in an example
JSON object -- LangChain tries to interpret them as template variables and fails.


In [ ]:
# WRONG -- looks like it should have ONE variable ({question}), but the JSON example
# also uses curly braces, which get misread as MORE variables.
try:
    broken_prompt = ChatPromptTemplate.from_messages([
        ("system", 'Respond in this format: {"name": "John", "age": 21}'),
        ("human", "{question}"),
    ])
    broken_prompt.invoke({"question": "What's your name?"})
except KeyError as e:
    print(f"Failed as expected: {e}")


In [ ]:
# RIGHT -- escape literal braces by doubling them: { becomes {{, } becomes }}
fixed_prompt = ChatPromptTemplate.from_messages([
    ("system", 'Respond in this format: {{"name": "John", "age": 21}}'),
    ("human", "{question}"),
])
result = fixed_prompt.invoke({"question": "What's your name?"})
print(result.to_messages())


## 4.4 `MessagesPlaceholder`: Injecting an Entire History as One Variable

Sometimes a template needs to slot in an entire LIST of prior messages, not just a string.


In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import AIMessage

history_aware_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder("history"),
    ("human", "{new_question}"),
])

prior_history = [
    HumanMessage("My name is Rina."),
    AIMessage("Nice to meet you, Rina!"),
]

result = history_aware_prompt.invoke({"history": prior_history, "new_question": "What's my name?"})
response = model.invoke(result.to_messages())
print(response.content)


## 4.5 Coercion Rules: What Counts as a Valid Message

LangChain accepts several "message-like" shapes and auto-converts them. A dict missing
`content`, or with an unrecognized `role`, triggers `MESSAGE_COERCION_FAILURE`.


In [ ]:
valid_shapes = [
    HumanMessage("A real message object"),
    {"role": "user", "content": "An OpenAI-style dict"},
    ("human", "A role/content tuple"),
    "A plain string, auto-converted to HumanMessage",
]
response = model.invoke([valid_shapes[0]])
print("All four shapes above are valid inputs -- LangChain coerces them consistently.")


---
# What's Next

```mermaid
graph LR
    A[Part 4: Templates - done] --> B[Part 5: Structured Output]
    B --> C[Part 6: Tools]
    C --> D[Part 7: Agents]
```

- **Part 5 — Structured Output:** forcing a model's reply into a guaranteed, validated shape
  (you got a preview of `with_structured_output()` in your own live exploration).
- **Part 6 — Tools:** giving a model real capabilities, properly, beyond the manual loop shown
  in section 3.7.
- **Part 7 — Agents:** `create_agent` automates the request -> execute -> respond loop you just
  built by hand.

## Full Revision Self-Check

1. Name the four Lang-family products and what each is for.
2. What's the difference between `runtime.state` and `runtime.context`? *(preview -- covered fully in Part 6)*
3. Why does a `ChatPromptTemplate` beat an f-string?
4. What causes `INVALID_PROMPT_INPUT`, and how do you fix it?
5. What's the difference between `.batch()` and `.batch_as_completed()`?
6. What does `ToolMessage.artifact` do that `.content` doesn't?
